# Lumen Cloud: Wan2GP AI video worker (free Colab GPU)

This notebook turns a free Colab GPU into the **AI video** source for Lumen Cloud. It runs
[Wan2GP](https://github.com/deepbeepmeep/Wan2GP) by DeepBeepMeep and generates a short clip for every
scene you queue from the studio (*Visual → AI video → Queue AI video*).

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (or better).
2. In Vercel, add `WAN2GP_WORKER_SECRET` (32+ random characters, e.g. `openssl rand -hex 32`) and redeploy.
3. In Colab, open the 🔑 **Secrets** panel and add `LUMEN_URL` (your studio's https URL) and
   `WAN2GP_WORKER_SECRET` (the same value), with notebook access switched on.

Then *Runtime → Run all*. The first run downloads the model (about 3 GB for the default 1.3B model).
Leave the tab open while clips are generated. Stopping it is safe: an unfinished clip is re-queued
after 45 minutes, and the scene keeps its current image in the meantime.

In [ ]:
#@title 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 2. Install Wan2GP (about 5 minutes)
%cd /content
!git clone --depth 1 https://github.com/deepbeepmeep/Wan2GP
%cd /content/Wan2GP
# Colab already ships CUDA PyTorch; install the rest of Wan2GP's requirements on top of it.
!pip install -q -r requirements.txt
!wget -q -O /content/wan2gp_worker.py https://raw.githubusercontent.com/gyuv/youtube-ai-agent/main/cloud/colab/wan2gp_worker.py

In [ ]:
#@title 3. Settings
#@markdown `t2v_1.3B` (Wan 2.1, 1.3B) fits a free T4. Clips are 480p, up to 5 seconds; Lumen loops
#@markdown them to cover longer narration and scales them to 1080p.
MODEL = "t2v_1.3B"  #@param ["t2v_1.3B", "t2v", "t2v_2_2"] {allow-input: true}
#@markdown Leave empty for text-to-video. Set to e.g. `i2v` to animate each scene's existing image
#@markdown instead (14B model: needs a bigger GPU and RAM than a free T4 runtime has).
I2V_MODEL = ""  #@param {type:"string"}
#@markdown Fewer steps = faster, lower quality. 20 is a good balance on a T4.
STEPS = 20  #@param {type:"slider", min:8, max:50, step:1}
MAX_FRAMES = 81  #@param {type:"integer"}

import os
from google.colab import userdata
os.environ["LUMEN_URL"] = userdata.get("LUMEN_URL")
os.environ["WAN2GP_WORKER_SECRET"] = userdata.get("WAN2GP_WORKER_SECRET")

In [ ]:
#@title 4. Run the worker (keep this running)
!python /content/wan2gp_worker.py --wan2gp-dir /content/Wan2GP --model "{MODEL}" --i2v-model "{I2V_MODEL}" --steps {STEPS} --max-frames {MAX_FRAMES}